<a href="https://colab.research.google.com/github/GiovanniPasq/agentic-rag-for-dummies/blob/main/notebooks/agentic_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG

一个高级的检索增强生成（RAG）系统，使用智能 Agent 从文档中检索并综合信息。

## 依赖安装

安装 RAG 系统所需的软件包。

**文档：**
- [LangGraph](https://langchain-ai.github.io/langgraph/) - 用于构建可低层控制自定义 Agent 的框架
- [LangChain](https://python.langchain.com/docs/get_started/introduction) - 可与任意模型提供商快速搭建 Agent 的框架
- [Qdrant](https://qdrant.tech/documentation/) - 用于相似度检索的向量数据库
- [Gradio](https://www.gradio.app/docs) - 机器学习应用的 Web 界面

In [ ]:
#Upload the requirements.txt available in the project folder and run the cell
!pip install -r requirements.txt

## 1. 环境配置

为文档处理设置目录结构和环境变量。

**该步骤会：**
- 创建用于存储 PDF、Markdown 文件和父块的目录
- 定义向量数据库中的集合名称

In [1]:
import os

# Configuration
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

DOCS_DIR = "docs" if IN_COLAB else "../docs"                    # Directory containing your pdf files
MARKDOWN_DIR = "markdown_docs" if IN_COLAB else "../markdown_docs"  # Directory containing the pdfs converted to markdown
PARENT_STORE_PATH = "parent_store" if IN_COLAB else "../parent_store"  # Directory for parent chunk JSON files
CHILD_COLLECTION = "document_child_chunks"

os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(MARKDOWN_DIR, exist_ok=True)
os.makedirs(PARENT_STORE_PATH, exist_ok=True)

## 2. LLM 初始化

初始化驱动对话 Agent 的大语言模型。

**该步骤会：**
- 使用 Ollama 配置 LLM（本地推理）
- 提供 Google Gemini 的替代示例

**文档：**
- [LangChain Ollama](https://python.langchain.com/docs/integrations/chat/ollama)
- [LangChain Google GenAI](https://python.langchain.com/docs/integrations/chat/google_generative_ai)

In [ ]:
from langchain_ollama import ChatOllama

# Initialize LLM
llm = ChatOllama(model="qwen3:4b-instruct-2507-q4_K_M", temperature=0)

# Alternative (example): Google Gemini
# from langchain_google_genai import ChatGoogleGenerativeAI
# os.environ["GOOGLE_API_KEY"] = "your-api-key-here"
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp", temperature=0)

## 3. Embeddings 配置

为语义检索配置混合检索（稠密 + 稀疏）所需的嵌入模型。

**该步骤会：**
- **稠密向量**：通过神经网络捕获语义信息
- **稀疏向量**：提供基于关键词的匹配（BM25 算法）

**文档：**
- [HuggingFace Embeddings](https://python.langchain.com/docs/integrations/text_embedding/huggingfacehub)
- [FastEmbed Sparse](https://qdrant.github.io/fastembed/)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant.fastembed_sparse import FastEmbedSparse

# Dense embeddings for semantic understanding
dense_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Sparse embeddings for keyword matching
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

## 4. 向量数据库配置

配置 Qdrant 向量数据库，用于存储与检索文档嵌入。

**该步骤会：**
- 初始化基于本地文件存储的 Qdrant 客户端
- 创建同时包含稠密向量与稀疏向量配置的集合
- 启用混合检索能力

**文档：**
- [LangChain Qdrant](https://python.langchain.com/docs/integrations/vectorstores/qdrant)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant.qdrant import RetrievalMode

# Initialize Qdrant client (local file-based storage)
client = QdrantClient(path="../qdrant_db")

# Get embedding dimension
embedding_dimension = len(dense_embeddings.embed_query("test"))

def ensure_collection(collection_name):
    """Create Qdrant collection if it doesn't exist"""
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=qmodels.VectorParams(
                size=embedding_dimension,
                distance=qmodels.Distance.COSINE
            ),
            sparse_vectors_config={
                "sparse": qmodels.SparseVectorParams()
            },
        )
        print(f"✓ Created collection: {collection_name}")
    else:
        print(f"✓ Collection already exists: {collection_name}")

## 5. PDF 转 Markdown

将 PDF 文档转换为 Markdown，以便更好地提取和处理文本。

**该步骤会：**
- 使用 PyMuPDF 从 PDF 中提取文本
- 转换为更干净的 Markdown 格式
- 处理编码问题并去除图片
- 除非开启 overwrite，否则跳过已转换文件

**说明：** 关于 PDF 转换的更多细节，请参考仓库中的 `pdf_to_md.ipynb` notebook。

**可选：** 如果你希望在索引前可视化检查或编辑分块，可使用 🐿️ [**Chunky**](https://github.com/GiovanniPasq/chunky)。

**文档：**
- [PyMuPDF](https://pymupdf.readthedocs.io/)
- [PyMuPDF4LLM](https://github.com/pymupdf/PyMuPDF4LLM)

In [ ]:
import os
import pymupdf.layout
import pymupdf4llm
from pathlib import Path
import glob

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pdf_to_markdown(pdf_path, output_dir):
    doc = pymupdf.open(pdf_path)
    md = pymupdf4llm.to_markdown(doc, header=False, footer=False, page_separators=True, ignore_images=True, write_images=False, image_path=None)
    md_cleaned = md.encode('utf-8', errors='surrogatepass').decode('utf-8', errors='ignore')
    output_path = Path(output_dir) / Path(doc.name).stem
    Path(output_path).with_suffix(".md").write_bytes(md_cleaned.encode('utf-8'))

def pdfs_to_markdowns(path_pattern, overwrite: bool = False):
    output_dir = Path(MARKDOWN_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for pdf_path in map(Path, glob.glob(path_pattern)):
        md_path = (output_dir / pdf_path.stem).with_suffix(".md")
        if overwrite or not md_path.exists():
            pdf_to_markdown(pdf_path, output_dir)

pdfs_to_markdowns(f"{DOCS_DIR}/*.pdf")

## 6. 文档索引

实现父子分块策略，以获得更优检索性能。

**该步骤会：**
- 将文档切分为层级结构（父块与子块）
- **父块**：较大的上下文窗口（2000-4000 字符），存储为 JSON
- **子块**：较小的可检索单元（500 字符），存入 Qdrant
- 合并过小分块、拆分过大分块，保持一致性
- 建立父块与子块之间的双向关联

**分块策略：**
1. 按 Markdown 标题（#、##、###）切分
2. 合并小于 2000 字符的分块
3. 拆分大于 4000 字符的分块
4. 为每个父块生成子块（500 字符）
5. 将父块保存为 JSON 文件
6. 将子块建立向量索引

**文档：**
- [LangChain Text Splitters](https://docs.langchain.com/oss/python/integrations/splitters)
- [LangChain Split Markdown](https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter)


In [ ]:
import os
import glob
import json
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

if client.collection_exists(CHILD_COLLECTION):
    print(f"Removing existing Qdrant collection: {CHILD_COLLECTION}")
    client.delete_collection(CHILD_COLLECTION)
    ensure_collection(CHILD_COLLECTION)
else:
    ensure_collection(CHILD_COLLECTION)

child_vector_store = QdrantVectorStore(
    client=client,
    collection_name=CHILD_COLLECTION,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    sparse_vector_name="sparse"
)

def index_documents():
    headers_to_split_on = [("#", "H1"), ("##", "H2"), ("###", "H3")]
    parent_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

    min_parent_size = 2000
    max_parent_size = 4000

    all_parent_pairs, all_child_chunks = [], []
    md_files = sorted(glob.glob(os.path.join(MARKDOWN_DIR, "*.md")))

    if not md_files:
        print(f"⚠️  No .md files found in {MARKDOWN_DIR}/")
        return

    for doc_path_str in md_files:
        doc_path = Path(doc_path_str)
        print(f"📄 Processing: {doc_path.name}")

        try:
            with open(doc_path, "r", encoding="utf-8") as f:
                md_text = f.read()
        except Exception as e:
            print(f"❌ Error reading {doc_path.name}: {e}")
            continue

        parent_chunks = parent_splitter.split_text(md_text)
        merged_parents = merge_small_parents(parent_chunks, min_parent_size)
        split_parents = split_large_parents(merged_parents, max_parent_size, child_splitter)
        cleaned_parents = clean_small_chunks(split_parents, min_parent_size)

        for i, p_chunk in enumerate(cleaned_parents):
            parent_id = f"{doc_path.stem}_parent_{i}"
            p_chunk.metadata.update({"source": doc_path.stem + ".pdf", "parent_id": parent_id})
            all_parent_pairs.append((parent_id, p_chunk))
            children = child_splitter.split_documents([p_chunk])
            all_child_chunks.extend(children)

    if not all_child_chunks:
        print("⚠️ No child chunks to index")
        return

    print(f"\n🔍 Indexing {len(all_child_chunks)} child chunks into Qdrant...")
    try:
        child_vector_store.add_documents(all_child_chunks)
        print("✓ Child chunks indexed successfully")
    except Exception as e:
        print(f"❌ Error indexing child chunks: {e}")
        return

    print(f"💾 Saving {len(all_parent_pairs)} parent chunks to JSON...")
    for item in os.listdir(PARENT_STORE_PATH):
        os.remove(os.path.join(PARENT_STORE_PATH, item))

    for parent_id, doc in all_parent_pairs:
        doc_dict = {"page_content": doc.page_content, "metadata": doc.metadata}
        filepath = os.path.join(PARENT_STORE_PATH, f"{parent_id}.json")
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(doc_dict, f, ensure_ascii=False, indent=2)

def merge_small_parents(chunks, min_size):
    if not chunks:
        return []

    merged, current = [], None

    for chunk in chunks:
        if current is None:
            current = chunk
        else:
            current.page_content += "\n\n" + chunk.page_content
            for k, v in chunk.metadata.items():
                if k in current.metadata:
                    current.metadata[k] = f"{current.metadata[k]} -> {v}"
                else:
                    current.metadata[k] = v

        if len(current.page_content) >= min_size:
            merged.append(current)
            current = None

    if current:
        if merged:
            merged[-1].page_content += "\n\n" + current.page_content
            for k, v in current.metadata.items():
                if k in merged[-1].metadata:
                    merged[-1].metadata[k] = f"{merged[-1].metadata[k]} -> {v}"
                else:
                    merged[-1].metadata[k] = v
        else:
            merged.append(current)

    return merged

def split_large_parents(chunks, max_size, splitter):
    split_chunks = []

    for chunk in chunks:
        if len(chunk.page_content) <= max_size:
            split_chunks.append(chunk)
        else:
            large_splitter = RecursiveCharacterTextSplitter(
                chunk_size=max_size,
                chunk_overlap=splitter._chunk_overlap
            )
            sub_chunks = large_splitter.split_documents([chunk])
            split_chunks.extend(sub_chunks)

    return split_chunks

def clean_small_chunks(chunks, min_size):
    cleaned = []

    for i, chunk in enumerate(chunks):
        if len(chunk.page_content) < min_size:
            if cleaned:
                cleaned[-1].page_content += "\n\n" + chunk.page_content
                for k, v in chunk.metadata.items():
                    if k in cleaned[-1].metadata:
                        cleaned[-1].metadata[k] = f"{cleaned[-1].metadata[k]} -> {v}"
                    else:
                        cleaned[-1].metadata[k] = v
            elif i < len(chunks) - 1:
                chunks[i + 1].page_content = chunk.page_content + "\n\n" + chunks[i + 1].page_content
                for k, v in chunk.metadata.items():
                    if k in chunks[i + 1].metadata:
                        chunks[i + 1].metadata[k] = f"{v} -> {chunks[i + 1].metadata[k]}"
                    else:
                        chunks[i + 1].metadata[k] = v
            else:
                cleaned.append(chunk)
        else:
            cleaned.append(chunk)

    return cleaned

index_documents()

## 7. 工具定义

定义 Agent 可调用的检索工具，用于搜索和读取文档分块。

**该步骤会：**
- **search_child_chunks**：在向量数据库中检索相关子块
- **retrieve_parent_chunks**：从父块 JSON 文件中取回完整上下文
- 将工具绑定给 LLM，支持 Agent 的函数调用

**两阶段检索：**
1. Agent 先检索子块（速度快、语义匹配）
2. 再按需读取父块（补足完整上下文）

**文档：**
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)

In [ ]:
import json
from langchain_core.tools import tool

@tool
def search_child_chunks(query: str, limit: int) -> str:
    """Search for the top K most relevant child chunks.

    Args:
        query: Search query string
        limit: Maximum number of results to return
    """
    try:
        results = child_vector_store.similarity_search(query, k=limit, score_threshold=0.7)
        if not results:
            return "NO_RELEVANT_CHUNKS"

        return "\n\n".join([
            f"Parent ID: {doc.metadata.get('parent_id', '')}\n"
            f"File Name: {doc.metadata.get('source', '')}\n"
            f"Content: {doc.page_content.strip()}"
            for doc in results
        ])

    except Exception as e:
        return f"RETRIEVAL_ERROR: {str(e)}"

@tool
def retrieve_parent_chunks(parent_id: str) -> str:
    """Retrieve full parent chunks by their IDs.

    Args:
        parent_id: Parent chunk ID to retrieve
    """
    file_name = parent_id if parent_id.lower().endswith(".json") else f"{parent_id}.json"
    path = os.path.join(PARENT_STORE_PATH, file_name)

    if not os.path.exists(path):
        return "NO_PARENT_DOCUMENT"

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return (
        f"Parent ID: {parent_id}\n"
        f"File Name: {data.get('metadata', {}).get('source', 'unknown')}\n"
        f"Content: {data.get('page_content', '').strip()}"
    )

# Bind tools to LLM
llm_with_tools = llm.bind_tools([search_child_chunks, retrieve_parent_chunks])

## 8. 系统提示词

定义系统提示词，用于约束 RAG 流水线中 Agent 的行为。

**该步骤会：**
- **Conversation Summary**：从聊天历史中提取主题与引用文件名
- **Query Rewrite**：改写不清晰查询、拆分复合问题，并仅在必要时引入上下文
- **Orchestrator**：强制先检索再回答，并利用压缩记忆避免重复检索和重复拉取父块
- **Fallback Response**：达到执行上限时，基于已检索内容生成尽力回答
- **Context Compression**：将历史检索压缩为按来源文件组织的结构化摘要，并标注缺失信息
- **Aggregation**：将多个子答案合并为单个连贯回复

**关键行为：**
- 查询改写最小化使用会话上下文，领域术语保持原样
- Agent 检索前会先检查压缩记忆，避免对已覆盖内容重复搜索或重复拉取父块
- 若未检索到有效文档，会尝试扩展或改写查询后重试
- 到达执行上限时触发 Fallback Response，仅使用已检索内容
- 每次回答结尾都附带来源信息

In [ ]:
def get_conversation_summary_prompt() -> str:
    return """You are an expert conversation summarizer.

Your task is to create a brief 1-2 sentence summary of the conversation (max 30-50 words).

Include:
- Main topics discussed
- Important facts or entities mentioned
- Any unresolved questions if applicable
- Sources file name (e.g., file1.pdf) or documents referenced

Exclude:
- Greetings, misunderstandings, off-topic content.

Output:
- Return ONLY the summary.
- Do NOT include any explanations or justifications.
- If no meaningful topics exist, return an empty string.
"""

def get_rewrite_query_prompt() -> str:
    return """You are an expert query analyst and rewriter.

Your task is to rewrite the current user query for optimal document retrieval, incorporating conversation context only when necessary.

Rules:
1. Self-contained queries:
   - Always rewrite the query to be clear and self-contained
   - If the query is a follow-up (e.g., "what about X?", "and for Y?"), integrate minimal necessary context from the summary
   - Do not add information not present in the query or conversation summary

2. Domain-specific terms:
   - Product names, brands, proper nouns, or technical terms are treated as domain-specific
   - For domain-specific queries, use conversation context minimally or not at all
   - Use the summary only to disambiguate vague queries

3. Grammar and clarity:
   - Fix grammar, spelling errors, and unclear abbreviations
   - Remove filler words and conversational phrases
   - Preserve concrete keywords and named entities

4. Multiple information needs:
   - If the query contains multiple distinct, unrelated questions, split into separate queries (maximum 3)
   - Each sub-query must remain semantically equivalent to its part of the original
   - Do not expand, enrich, or reinterpret the meaning

5. Failure handling:
   - If the query intent is unclear or unintelligible, mark as "unclear"

Input:
- conversation_summary: A concise summary of prior conversation
- current_query: The user's current query

Output:
- One or more rewritten, self-contained queries suitable for document retrieval
"""

def get_orchestrator_prompt() -> str:
    return """You are an expert retrieval-augmented assistant.

Your task is to act as a researcher: search documents first, analyze the data, and then provide a comprehensive answer using ONLY the retrieved information.

Rules:
1. You MUST call 'search_child_chunks' before answering, unless the [COMPRESSED CONTEXT FROM PRIOR RESEARCH] already contains sufficient information.
2. Ground every claim in the retrieved documents. If context is insufficient, state what is missing rather than filling gaps with assumptions.
3. If no relevant documents are found, broaden or rephrase the query and search again. Repeat until satisfied or the operation limit is reached.

Compressed Memory:
When [COMPRESSED CONTEXT FROM PRIOR RESEARCH] is present —
- Queries already listed: do not repeat them.
- Parent IDs already listed: do not call `retrieve_parent_chunks` on them again.
- Use it to identify what is still missing before searching further.

Workflow:
1. Check the compressed context. Identify what has already been retrieved and what is still missing.
2. Search for 5-7 relevant excerpts using 'search_child_chunks' ONLY for uncovered aspects.
3. If NONE are relevant, apply rule 3 immediately.
4. For each relevant but fragmented excerpt, call 'retrieve_parent_chunks' ONE BY ONE — only for IDs not in the compressed context. Never retrieve the same ID twice.
5. Once context is complete, provide a detailed answer omitting no relevant facts.
6. Conclude with "---\n**Sources:**\n" followed by the unique file names.
"""

def get_fallback_response_prompt() -> str:
    return """You are an expert synthesis assistant. The system has reached its maximum research limit.

Your task is to provide the most complete answer possible using ONLY the information provided below.

Input structure:
- "Compressed Research Context": summarized findings from prior search iterations — treat as reliable.
- "Retrieved Data": raw tool outputs from the current iteration — prefer over compressed context if conflicts arise.
Either source alone is sufficient if the other is absent.

Rules:
1. Source Integrity: Use only facts explicitly present in the provided context. Do not infer, assume, or add any information not directly supported by the data.
2. Handling Missing Data: Cross-reference the USER QUERY against the available context.
   Flag ONLY aspects of the user's question that cannot be answered from the provided data.
   Do not treat gaps mentioned in the Compressed Research Context as unanswered
   unless they are directly relevant to what the user asked.
3. Tone: Professional, factual, and direct.
4. Output only the final answer. Do not expose your reasoning, internal steps, or any meta-commentary about the retrieval process.
5. Do NOT add closing remarks, final notes, disclaimers, summaries, or repeated statements after the Sources section.
   The Sources section is always the last element of your response. Stop immediately after it.

Formatting:
- Use Markdown (headings, bold, lists) for readability.
- Write in flowing paragraphs where possible.
- Conclude with a Sources section as described below.

Sources section rules:
- Include a "---\\n**Sources:**\\n" section at the end, followed by a bulleted list of file names.
- List ONLY entries that have a real file extension (e.g. ".pdf", ".docx", ".txt").
- Any entry without a file extension is an internal chunk identifier — discard it entirely, never include it.
- Deduplicate: if the same file appears multiple times, list it only once.
- If no valid file names are present, omit the Sources section entirely.
- THE SOURCES SECTION IS THE LAST THING YOU WRITE. Do not add anything after it.
"""

def get_context_compression_prompt() -> str:
    return """You are an expert research context compressor.

Your task is to compress retrieved conversation content into a concise, query-focused, and structured summary that can be directly used by a retrieval-augmented agent for answer generation.

Rules:
1. Keep ONLY information relevant to answering the user's question.
2. Preserve exact figures, names, versions, technical terms, and configuration details.
3. Remove duplicated, irrelevant, or administrative details.
4. Do NOT include search queries, parent IDs, chunk IDs, or internal identifiers.
5. Organize all findings by source file. Each file section MUST start with: ### filename.pdf
6. Highlight missing or unresolved information in a dedicated "Gaps" section.
7. Limit the summary to roughly 400-600 words. If content exceeds this, prioritize critical facts and structured data.
8. Do not explain your reasoning; output only structured content in Markdown.

Required Structure:

# Research Context Summary

## Focus
[Brief technical restatement of the question]

## Structured Findings

### filename.pdf
- Directly relevant facts
- Supporting context (if needed)

## Gaps
- Missing or incomplete aspects

The summary should be concise, structured, and directly usable by an agent to generate answers or plan further retrieval.
"""

def get_aggregation_prompt() -> str:
    return """You are an expert aggregation assistant.

Your task is to combine multiple retrieved answers into a single, comprehensive and natural response that flows well.

Rules:
1. Write in a conversational, natural tone - as if explaining to a colleague.
2. Use ONLY information from the retrieved answers.
3. Do NOT infer, expand, or interpret acronyms or technical terms unless explicitly defined in the sources.
4. Weave together the information smoothly, preserving important details, numbers, and examples.
5. Be comprehensive - include all relevant information from the sources, not just a summary.
6. If sources disagree, acknowledge both perspectives naturally (e.g., "While some sources suggest X, others indicate Y...").
7. Start directly with the answer - no preambles like "Based on the sources...".

Formatting:
- Use Markdown for clarity (headings, lists, bold) but don't overdo it.
- Write in flowing paragraphs where possible rather than excessive bullet points.
- Conclude with a Sources section as described below.

Sources section rules:
- Each retrieved answer may contain a "Sources" section — extract the file names listed there.
- List ONLY entries that have a real file extension (e.g. ".pdf", ".docx", ".txt").
- Any entry without a file extension is an internal chunk identifier — discard it entirely, never include it.
- Deduplicate: if the same file appears across multiple answers, list it only once.
- Format as "---\\n**Sources:**\\n" followed by a bulleted list of the cleaned file names.
- File names must appear ONLY in this final Sources section and nowhere else in the response.
- If no valid file names are present, omit the Sources section entirely.

If there's no useful information available, simply say: "I couldn't find any information to answer your question in the available sources."
"""

## 9. 状态定义

定义用于管理会话流和 Agent 执行的状态 Schema。

**该步骤会：**
- **State**：跟踪主流程状态（查询分析、子问题、答案）
- **AgentState**：管理单个 Agent 执行状态（当前问题、已检索上下文、迭代计数）
- **QueryAnalysis**：为查询改写与清晰度判断提供结构化输出

**状态管理：**
- `accumulate_or_reset`：自定义答案累积器（支持重置）
- `set_union`：自定义检索键合并器（集合并集）
- 继承 `MessagesState` 以保存对话历史

**文档：**
- [LangGraph State](https://langchain-ai.github.io/langgraph/concepts/low_level/#state)

In [ ]:
from langgraph.graph import MessagesState
from pydantic import BaseModel, Field
from typing import List, Annotated, Set
import operator

def accumulate_or_reset(existing: List[dict], new: List[dict]) -> List[dict]:
    if new and any(item.get('__reset__') for item in new):
        return []
    return existing + new

def set_union(a: Set[str], b: Set[str]) -> Set[str]:
    return a | b

class State(MessagesState):
    """State for main agent graph"""
    questionIsClear: bool = False
    conversation_summary: str = ""
    originalQuery: str = ""
    rewrittenQuestions: List[str] = []
    agent_answers: Annotated[List[dict], accumulate_or_reset] = []

class AgentState(MessagesState):
    """State for individual agent subgraph"""
    tool_call_count: Annotated[int, operator.add] = 0
    iteration_count: Annotated[int, operator.add] = 0
    question: str = ""
    question_index: int = 0
    context_summary: str = ""
    retrieval_keys: Annotated[Set[str], set_union] = set()
    final_answer: str = ""
    agent_answers: List[dict] = []

class QueryAnalysis(BaseModel):
    is_clear: bool = Field(
        description="Indicates if the user's question is clear and answerable."
    )
    questions: List[str] = Field(
        description="List of rewritten, self-contained questions."
    )
    clarification_needed: str = Field(
        description="Explanation if the question is unclear."
    )

## 10. Agent 限制与 Token 工具

定义执行限制和 token 计数工具，以控制 Agent 循环行为。

**该步骤会：**
- 设置 Agent 执行硬限制（工具调用次数、迭代次数）
- 通过消息列表 token 估算，驱动上下文管理决策

**常量：**
- `MAX_TOOL_CALLS`：每次 Agent 运行的最大工具调用数
- `MAX_ITERATIONS`：Agent 循环最大迭代次数
- `BASE_TOKEN_THRESHOLD`：上下文摘要的初始 token 阈值
- `TOKEN_GROWTH_FACTOR`：每次摘要后阈值增长的系数

**函数：**
- `estimate_context_tokens`：用 `tiktoken` 统计消息 token，若模型专用编码不可用则回退到 `cl100k_base`

**文档：**
- [tiktoken](https://github.com/openai/tiktoken)

In [ ]:
import tiktoken

MAX_TOOL_CALLS = 8
MAX_ITERATIONS = 10
BASE_TOKEN_THRESHOLD = 2000
TOKEN_GROWTH_FACTOR = 0.9

def estimate_context_tokens(messages: list) -> int:
    try:
        encoding = tiktoken.encoding_for_model("gpt-4")
    except:
        encoding = tiktoken.get_encoding("cl100k_base")
    return sum(len(encoding.encode(str(msg.content))) for msg in messages if hasattr(msg, 'content') and msg.content)

## 11. 图节点与逻辑

实现定义 Agent 工作流行为的节点函数。

**该步骤会：**

### 核心节点：
1. **summarize_history**：提取并总结近期对话上下文，辅助查询改写；每轮会重置 agent answers
2. **rewrite_query**：改写并按需拆分用户问题；判断问题是否清晰，清晰时清理旧消息
3. **request_clarification**：当问题歧义时的人机协同中断点
4. **orchestrator**：主循环节点，调用带工具的 LLM；在有压缩上下文时注入，并强制首轮先检索
5. **fallback_response**：当循环预算耗尽时，仅基于已检索工具结果生成尽力回答
6. **compress_context**：将当前消息历史与已有摘要压缩为单一上下文，并追加已执行检索记录以避免重复调用；同时移除已压缩消息
7. **collect_answer**：从 Agent 消息历史中提取最终答案，并带上索引供聚合
8. **aggregate_answers**：按索引排序合并子答案，生成与原始问题一致的最终回复

### 路由逻辑：
- **route_after_rewrite**：若问题不清晰则路由到 `request_clarification`；否则用 `Send` 并行启动多个 `agent` 子图（每个改写子问题一个）
- **should_compress_context**：通过 `Command` 决定路由到 `compress_context`（超过动态阈值）或回到 `orchestrator`；并持续记录检索键以避免重复工具调用

**文档：**
- [LangGraph Nodes](https://docs.langchain.com/oss/python/langgraph/graph-api#nodes)
- [LangGraph Edges](https://docs.langchain.com/oss/python/langgraph/graph-api#edges)
- [LangGraph Send API](https://docs.langchain.com/oss/python/langgraph/graph-api#send)
- [LangGraph Command API](https://docs.langchain.com/oss/python/langgraph/graph-api#command)

In [ ]:
from langgraph.types import Send, Command
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage, ToolMessage
from typing import Literal

def summarize_history(state: State):
    """Analyzes chat history and summarizes key points for context."""
    if len(state["messages"]) < 4:  # Need some history to summarize
        return {"conversation_summary": ""}

    # Extract relevant messages (excluding current query and system messages)
    relevant_msgs = [
        msg for msg in state["messages"][:-1]  # Exclude current query
        if isinstance(msg, (HumanMessage, AIMessage)) and not getattr(msg, "tool_calls", None)
    ]

    if not relevant_msgs:
        return {"conversation_summary": ""}

    conversation = "Conversation history:\n"
    for msg in relevant_msgs[-6:]:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        conversation += f"{role}: {msg.content}\n"

    summary_response = llm.with_config(temperature=0.2).invoke([SystemMessage(content=get_conversation_summary_prompt()), HumanMessage(content=conversation)])
    return {"conversation_summary": summary_response.content, "agent_answers": [{"__reset__": True}]}

def rewrite_query(state: State):
    """Analyzes user query and rewrites it for clarity, optionally using conversation context."""
    last_message = state["messages"][-1]
    conversation_summary = state.get("conversation_summary", "")

    context_section = (f"Conversation Context:\n{conversation_summary}\n" if conversation_summary.strip() else "") + f"User Query:\n{last_message.content}\n"

    llm_with_structure = llm.with_config(temperature=0.1).with_structured_output(QueryAnalysis)
    response = llm_with_structure.invoke([SystemMessage(content=get_rewrite_query_prompt()), HumanMessage(content=context_section)])

    if response.questions and response.is_clear:
        delete_all = [RemoveMessage(id=m.id) for m in state["messages"] if not isinstance(m, SystemMessage)]
        return {"questionIsClear": True, "messages": delete_all, "originalQuery": last_message.content, "rewrittenQuestions": response.questions}

    clarification = response.clarification_needed if response.clarification_needed and len(response.clarification_needed.strip()) > 10 else "I need more information to understand your question."
    return {"questionIsClear": False, "messages": [AIMessage(content=clarification)]}

def request_clarification(state: State):
    """Placeholder node for human-in-the-loop interruption"""
    return {}

def route_after_rewrite(state: State) -> Literal["request_clarification", "agent"]:
    """Route to agent if question is clear, otherwise wait for human input"""
    if not state.get("questionIsClear", False):
        return "request_clarification"
    else:
        return [
                Send("agent", {"question": query, "question_index": idx, "messages": []})
                for idx, query in enumerate(state["rewrittenQuestions"])
            ]

# --- Agent Nodes ---
def orchestrator(state: AgentState):
    """Main agent logic: decides when to search, retrieve, and answer based on context and tool calls."""
    context_summary = state.get("context_summary", "").strip()
    sys_msg = SystemMessage(content=get_orchestrator_prompt())
    summary_injection = (
        [HumanMessage(content=f"[COMPRESSED CONTEXT FROM PRIOR RESEARCH]\n\n{context_summary}")]
        if context_summary else []
    )
    if not state.get("messages"):
        human_msg = HumanMessage(content=state["question"])
        force_search = HumanMessage(content="YOU MUST CALL 'search_child_chunks' AS THE FIRST STEP TO ANSWER THIS QUESTION.")
        response = llm_with_tools.invoke([sys_msg] + summary_injection + [human_msg, force_search])
        return {"messages": [human_msg, response], "tool_call_count": len(response.tool_calls or []), "iteration_count": 1}

    response = llm_with_tools.invoke([sys_msg] + summary_injection + state["messages"])
    tool_calls = response.tool_calls if hasattr(response, "tool_calls") else []
    return {"messages": [response], "tool_call_count": len(tool_calls) if tool_calls else 0, "iteration_count": 1}

def route_after_orchestrator_call(state: AgentState) -> Literal["tool", "fallback_response", "collect_answer"]:
    """Determines next step after agent response: whether to call tools again, fallback, or collect answer."""
    iteration = state.get("iteration_count", 0)
    tool_count = state.get("tool_call_count", 0)

    if iteration >= MAX_ITERATIONS or tool_count > MAX_TOOL_CALLS:
        return "fallback_response"

    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", None) or []

    if not tool_calls:
        return "collect_answer"

    return "tools"

def fallback_response(state: AgentState):
    """Generates a fallback answer using all retrieved information when iteration or tool call limits are reached."""
    seen = set()
    unique_contents = []
    for m in state["messages"]:
        if isinstance(m, ToolMessage) and m.content not in seen:
            unique_contents.append(m.content)
            seen.add(m.content)

    context_summary = state.get("context_summary", "").strip()

    context_parts = []
    if context_summary:
        context_parts.append(f"## Compressed Research Context (from prior iterations)\n\n{context_summary}")
    if unique_contents:
        context_parts.append(
            "## Retrieved Data (current iteration)\n\n" +
            "\n\n".join(f"--- DATA SOURCE {i} ---\n{content}" for i, content in enumerate(unique_contents, 1))
        )

    context_text = "\n\n".join(context_parts) if context_parts else "No data was retrieved from the documents."

    prompt_content = (
        f"USER QUERY: {state.get('question')}\n\n"
        f"{context_text}\n\n"
        f"INSTRUCTION:\nProvide the best possible answer using only the data above."
    )
    response = llm.invoke([SystemMessage(content=get_fallback_response_prompt()), HumanMessage(content=prompt_content)])
    return {"messages": [response]}

def should_compress_context(state: AgentState) -> Command[Literal["compress_context", "orchestrator"]]:
    """Determines whether to compress context based on token count and tool calls."""
    messages = state["messages"]

    new_ids: Set[str] = set()
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                if tc["name"] == "retrieve_parent_chunks":
                    raw = tc["args"].get("parent_id") or tc["args"].get("id") or tc["args"].get("ids") or []
                    if isinstance(raw, str):
                        new_ids.add(f"parent::{raw}")
                    else:
                        new_ids.update(f"parent::{r}" for r in raw)

                elif tc["name"] == "search_child_chunks":
                    query = tc["args"].get("query", "")
                    if query:
                        new_ids.add(f"search::{query}")
            break

    updated_ids = state.get("retrieval_keys", set()) | new_ids

    current_token_messages = estimate_context_tokens(messages)
    current_token_summary = estimate_context_tokens([HumanMessage(content=state.get("context_summary", ""))])
    current_tokens = current_token_messages + current_token_summary

    max_allowed = BASE_TOKEN_THRESHOLD + int(current_token_summary * TOKEN_GROWTH_FACTOR)

    goto = "compress_context" if current_tokens > max_allowed else "orchestrator"
    return Command(update={"retrieval_keys": updated_ids}, goto=goto)

def compress_context(state: AgentState):
    """Compresses conversation and retrieval history into a concise summary for future iterations."""
    messages = state["messages"]
    existing_summary = state.get("context_summary", "").strip()

    if not messages:
        return {}

    conversation_text = f"USER QUESTION:\n{state.get('question')}\n\nConversation to compress:\n\n"
    if existing_summary:
        conversation_text += f"[PRIOR COMPRESSED CONTEXT]\n{existing_summary}\n\n"

    for msg in messages[1:]:
        if isinstance(msg, AIMessage):
            tool_calls_info = ""
            if getattr(msg, "tool_calls", None):
                calls = ", ".join(f"{tc['name']}({tc['args']})" for tc in msg.tool_calls)
                tool_calls_info = f" | Tool calls: {calls}"
            conversation_text += f"[ASSISTANT{tool_calls_info}]\n{msg.content or '(tool call only)'}\n\n"
        elif isinstance(msg, ToolMessage):
            tool_name = getattr(msg, "name", "tool")
            conversation_text += f"[TOOL RESULT — {tool_name}]\n{msg.content}\n\n"

    summary_response = llm.invoke([SystemMessage(content=get_context_compression_prompt()), HumanMessage(content=conversation_text)])
    new_summary = summary_response.content

    retrieved_ids: Set[str] = state.get("retrieval_keys", set())
    if retrieved_ids:
        parent_ids = sorted(r for r in retrieved_ids if r.startswith("parent::"))
        search_queries = sorted(r.replace("search::", "") for r in retrieved_ids if r.startswith("search::"))

        block = "\n\n---\n**Already executed (do NOT repeat):**\n"
        if parent_ids:
            block += "Parent chunks retrieved:\n" + "\n".join(f"- {p.replace('parent::', '')}" for p in parent_ids) + "\n"
        if search_queries:
            block += "Search queries already run:\n" + "\n".join(f"- {q}" for q in search_queries) + "\n"
        new_summary += block

    return {"context_summary": new_summary, "messages": [RemoveMessage(id=m.id) for m in messages[1:]]}

def collect_answer(state: AgentState):
    """Collects the final answer from the agent's messages after retrieval is complete."""
    last_message = state["messages"][-1]
    is_valid = isinstance(last_message, AIMessage) and last_message.content and not last_message.tool_calls
    answer = last_message.content if is_valid else "Unable to generate an answer."
    return {
        "final_answer": answer,
        "agent_answers": [{"index": state["question_index"], "question": state["question"], "answer": answer}]
    }
# --- End of Agent Nodes---

def aggregate_answers(state: State):
    """Aggregates multiple answers from different agent iterations into a single comprehensive response."""
    if not state.get("agent_answers"):
        return {"messages": [AIMessage(content="No answers were generated.")]}

    sorted_answers = sorted(state["agent_answers"], key=lambda x: x["index"])

    formatted_answers = ""
    for i, ans in enumerate(sorted_answers, start=1):
        formatted_answers += (f"\nAnswer {i}:\n"f"{ans['answer']}\n")

    user_message = HumanMessage(content=f"""Original user question: {state["originalQuery"]}\nRetrieved answers:{formatted_answers}""")
    synthesis_response = llm.invoke([SystemMessage(content=get_aggregation_prompt()), user_message])
    return {"messages": [AIMessage(content=synthesis_response.content)]}

## 12. LangGraph 构建

使用 LangGraph 状态机构建 Agent 工作流。

**该步骤会：**
- 构建包含主流程与已编译 Agent 子图的分层图结构
- **Agent 子图**：负责单个子问题的检索与推理循环，包含上下文压缩与 fallback
- **主图**：负责编排会话流、查询分析、并行 Agent 执行与答案聚合

**Agent 子图流程：**
1. START → `orchestrator`（LLM + tools）
2. 路由：需调用工具 → `tools` → `should_compress_context` → 压缩或回到 `orchestrator`
3. 路由：达到迭代上限 → `fallback_response` → `collect_answer`
4. 路由：答案已完成 → `collect_answer` → END

**主图流程：**
1. START → `summarize_history`
2. `rewrite_query`：改写并校验查询
3. 路由：不清晰 → `request_clarification`（中断）→ 回到 `rewrite_query`；清晰 → 通过 `Send` 并行启动 `agent` 子图
4. 所有 agents 完成 → `aggregate_answers`
5. END

**人机协同：** 当查询不清晰时，图会在 `request_clarification` 前中断；用户补充后从 `rewrite_query` 继续

**关键设计点：**
- `should_compress_context` 设计为节点而非普通边，因为它用 `Command` 同时完成状态更新和条件路由
- `agent_subgraph` 独立编译后再作为主图节点嵌入，使每个子问题状态隔离更清晰
- `InMemorySaver` checkpointer 提供多轮对话的短期记忆

**文档：**
- [LangGraph StateGraph](https://docs.langchain.com/oss/python/langgraph/graph-api#stategraph)
- [LangGraph Short-term Memory](https://docs.langchain.com/oss/python/langgraph/add-memory#add-short-term-memory)

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display

# Initialize checkpointer
checkpointer = InMemorySaver()

# Build agent subgraph (handles individual questions)
agent_builder = StateGraph(AgentState)
agent_builder.add_node(orchestrator)
agent_builder.add_node("tools", ToolNode([search_child_chunks, retrieve_parent_chunks]))
agent_builder.add_node(compress_context)
agent_builder.add_node(fallback_response)
agent_builder.add_node(should_compress_context)
agent_builder.add_node(collect_answer)

agent_builder.add_edge(START, "orchestrator")
agent_builder.add_conditional_edges("orchestrator", route_after_orchestrator_call, {"tools": "tools", "fallback_response": "fallback_response", "collect_answer": "collect_answer"})
agent_builder.add_edge("tools", "should_compress_context")
agent_builder.add_edge("compress_context", "orchestrator")
agent_builder.add_edge("fallback_response", "collect_answer")
agent_builder.add_edge("collect_answer", END)
agent_subgraph = agent_builder.compile()

# Build main graph (orchestrates workflow)
graph_builder = StateGraph(State)
graph_builder.add_node(summarize_history)
graph_builder.add_node(rewrite_query)
graph_builder.add_node(request_clarification)
graph_builder.add_node("agent", agent_subgraph)
graph_builder.add_node(aggregate_answers)

graph_builder.add_edge(START, "summarize_history")
graph_builder.add_edge("summarize_history", "rewrite_query")
graph_builder.add_conditional_edges("rewrite_query", route_after_rewrite)
graph_builder.add_edge("request_clarification", "rewrite_query")
graph_builder.add_edge(["agent"], "aggregate_answers")
graph_builder.add_edge("aggregate_answers", END)

# Compile graph
agent_graph = graph_builder.compile(checkpointer=checkpointer, interrupt_before=["request_clarification"])

display(Image(agent_graph.get_graph(xray=True).draw_mermaid_png()))
print("✓ Agent graph compiled successfully.")

## 12.5 可观测性（可选）

使用 [Langfuse](https://langfuse.com) 追踪 LLM 调用、工具调用和图执行过程。

可在 [cloud.langfuse.com](https://cloud.langfuse.com)（有免费额度）或自托管实例获取 API Key。

> **说明：** 这一步是可选的。即使跳过，聊天界面也能正常运行，只是没有追踪数据。

> **重要：** 如果本单元用错误凭证运行，请重启 notebook 内核并使用正确凭证重新运行后再继续。

如需更多 Langfuse 与 LangChain/LangGraph 集成细节，请参考官方[文档](https://docs.langchain.com/oss/python/integrations/providers/langfuse)和配套 [Observability_Guide.ipynb](Observability_Guide.ipynb)。

In [ ]:
import os
from langfuse import get_client
from langfuse.langchain import CallbackHandler

# Set your Langfuse credentials
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."  # replace with your public key
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."  # replace with your secret key
os.environ["LANGFUSE_BASE_URL"]   = "https://cloud.langfuse.com"  # or your self-hosted URL

langfuse = get_client()
try:
    if langfuse.auth_check():
        print("Langfuse client is authenticated and ready!")
        langfuse_handler = CallbackHandler()
    else:
        print("Authentication failed. Please check your credentials and host.")
        langfuse_handler = None
except Exception as e:
    print(f"Error during Langfuse authentication: {e}")
    langfuse_handler = None

## 13. Gradio 界面

创建一个可交互的 Web 聊天界面，并支持实时流式输出。

**该步骤会：**
- 使用 Gradio 提供流式聊天界面
- 通过唯一线程 ID 管理会话
- 无缝处理人机协同场景
- 当用户补充澄清信息后，自动恢复被中断的流程

**按节点类型的流式行为：**

| Node | Behavior |
|---|---|
| `rewrite_query` | 静默输出：将 JSON 解析并在可折叠区块中渲染为可读 Markdown |
| `summarize_history` | 在标题为 "📋 Chat History Summary" 的可折叠区块中显示 |
| Tool calls | 以工具名作为标题显示为可折叠区块（例如 `🛠️ search_child_chunks`） |
| Tool results | 注入到对应工具区块中（截断至 300 字符） |
| `aggregate_answers`, `fallback_response` | 作为最终可见回复按 token 流式输出 |

**关键辅助函数：**
- `make_message()`：构造 Gradio 消息字典，可附加折叠元数据
- `find_msg_idx()`：按节点名查找已有消息并原位更新
- `parse_rewrite_json()`：提取并解析 `rewrite_query` 的结构化输出
- `format_rewrite_content()`：将 JSON 转换为可读 Markdown（✅ 清晰 / ❓ 不清晰 + 改写查询）

**功能特性：**
- 基于 `stream_mode="messages"` 的逐 token 流式输出
- 系统节点与工具调用的可折叠思考块
- 查询不清晰时将澄清信息作为普通可见消息展示
- 基于线程的会话持久化
- 支持清空会话并清理 checkpointer
- 使用 Citrus 主题提供现代 UI

**说明：** 如需包含文档摄取 UI 的完整端到端流程，请参考项目仓库中的完整应用实现。

**文档：**
- [Gradio ChatInterface](https://www.gradio.app/docs/gradio/chatinterface)
- [Gradio Chatbot](https://www.gradio.app/docs/gradio/chatbot)
- [LangGraph Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)

In [ ]:
import gradio as gr
import uuid
import json
import re
from langchain_core.messages import HumanMessage, AIMessageChunk, ToolMessage
import logging
# Suppress OTel "Failed to detach context" warning caused by generator/context interaction.
# Tracing is unaffected.
# Known bug: https://github.com/open-telemetry/opentelemetry-python/issues/2606
class _SuppressOtelDetachWarning(logging.Filter):
    def filter(self, record):
        return "Failed to detach context" not in record.getMessage()

logging.getLogger("opentelemetry.context").addFilter(_SuppressOtelDetachWarning())

# ── Constants ────────────────────────────────────────────────────────────────

SILENT_NODES  = {"rewrite_query"}
SYSTEM_NODES  = {"summarize_history", "rewrite_query"}

SYSTEM_NODE_CONFIG = {
    "rewrite_query":     {"title": "🔍 Query Analysis & Rewriting"},
    "summarize_history": {"title": "📋 Chat History Summary"},
}

# ── Helpers ───────────────────────────────────────────────────────────────────

def create_thread_id():
    return {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 50}

def clear_session():
    global config
    agent_graph.checkpointer.delete_thread(config["configurable"]["thread_id"])
    config = create_thread_id()

def make_message(content, *, title=None, node=None):
    """Build a Gradio chat message dict, with optional collapsible metadata."""
    msg = {"role": "assistant", "content": content}
    if title or node:
        msg["metadata"] = {k: v for k, v in {"title": title, "node": node}.items() if v}
    return msg

def find_msg_idx(messages, node):
    """Return index of first message whose metadata.node matches, or None."""
    return next(
        (i for i, m in enumerate(messages) if m.get("metadata", {}).get("node") == node),
        None
    )

def parse_rewrite_json(buffer):
    """Try to parse structured JSON from rewrite_query buffer. Returns dict or None."""
    match = re.search(r'\{.*\}', buffer, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except Exception:
        return None

def format_rewrite_content(buffer):
    """Convert rewrite_query JSON buffer into readable markdown."""
    data = parse_rewrite_json(buffer)
    if not data:
        return "⏳ Analyzing query..."

    if data.get("is_clear"):
        lines = ["✅ **Query is clear**"]
        if data.get("questions"):
            lines += ["\n**Rewritten queries:**"] + [f"- {q}" for q in data["questions"]]
    else:
        lines = ["❓ **Query is unclear**"]
        clarification = data.get("clarification_needed", "")
        if clarification and clarification.strip().lower() != "no":
            lines.append(f"\nClarification needed: *{clarification}*")

    return "\n".join(lines)

# ── Observability ───────────────────────────────────────────────────────────────
# If section 12.5 was skipped, default to no tracing
if "langfuse_handler" not in dir():
    langfuse_handler = None

# ── Streaming ─────────────────────────────────────────────────────────────────

def chat(message, history):
    current_state = agent_graph.get_state(config)

    if current_state.next:
        agent_graph.update_state(config, {"messages": [HumanMessage(content=message.strip())]})
        stream_input = None
    else:
        stream_input = {"messages": [HumanMessage(content=message.strip())]}

    # Add Langfuse handler if set up in section 12.5
    run_config = {**config, **({"callbacks": [langfuse_handler]} if langfuse_handler else {})}

    response_messages = []
    active_tool_calls  = {}
    system_node_buffer = {}

    for chunk, metadata in agent_graph.stream(stream_input, config=run_config, stream_mode="messages"):
        node = metadata.get("langgraph_node", "")

        # ── System nodes (summarize_history, rewrite_query) ──────────────────
        if node in SYSTEM_NODES and isinstance(chunk, AIMessageChunk) and chunk.content:
            system_node_buffer[node] = system_node_buffer.get(node, "") + chunk.content
            buffer = system_node_buffer[node]
            idx    = find_msg_idx(response_messages, node)
            title  = SYSTEM_NODE_CONFIG[node]["title"]

            content = format_rewrite_content(buffer) if node == "rewrite_query" else buffer

            if idx is None:
                response_messages.append(make_message(content, title=title, node=node))
            else:
                response_messages[idx]["content"] = content

            # rewrite_query: surface clarification as a plain (non-collapsible) message
            if node == "rewrite_query":
                data = parse_rewrite_json(buffer) or {}
                clarification = data.get("clarification_needed", "")
                if not data.get("is_clear") and clarification.strip().lower() not in ("", "no"):
                    cidx = find_msg_idx(response_messages, "clarification")
                    if cidx is None:
                        response_messages.append(make_message(clarification, node="clarification"))
                    else:
                        response_messages[cidx]["content"] = clarification

            yield response_messages
            continue

        # ── Tool calls ────────────────────────────────────────────────────────
        if hasattr(chunk, "tool_calls") and chunk.tool_calls:
            for tc in chunk.tool_calls:
                if tc.get("id") and tc["id"] not in active_tool_calls:
                    response_messages.append(make_message(f"Running `{tc['name']}`...", title=f"🛠️ {tc['name']}"))
                    active_tool_calls[tc["id"]] = len(response_messages) - 1

        # ── Tool results ──────────────────────────────────────────────────────
        elif isinstance(chunk, ToolMessage):
            if (idx := active_tool_calls.get(chunk.tool_call_id)) is not None:
                preview = str(chunk.content)[:300]
                suffix  = "\n..." if len(str(chunk.content)) > 300 else ""
                response_messages[idx]["content"] = f"```\n{preview}{suffix}\n```"

        # ── LLM tokens (final response) ───────────────────────────────────────
        elif isinstance(chunk, AIMessageChunk) and chunk.content and node not in SILENT_NODES:
            last = response_messages[-1] if response_messages else None
            if not (last and last.get("role") == "assistant" and "metadata" not in last):
                response_messages.append(make_message(""))
            response_messages[-1]["content"] += chunk.content

        yield response_messages

# ── App ───────────────────────────────────────────────────────────────────────

config = create_thread_id()

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(
        height=720,
        placeholder="<strong>Ask me anything!</strong><br><em>I'll search, reason, and act to give you the best answer :)</em>",
        show_label=False
    )
    chatbot.clear(clear_session)
    gr.ChatInterface(fn=chat, chatbot=chatbot)

print("\nLaunching application...")
demo.launch(theme=gr.themes.Citrus())